In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
ROOT_DIR = Path("/Users/bmacedo/Desktop/final_WM")

In [3]:
INPUT_DATA_DIR = ROOT_DIR / "input_data"

In [4]:
# from /cbica/projects/abcd_qsiprep/meisler_abcd_paper/munge_data/merged_data_10282025_with_predictions.parquet
# white matter features
data = pd.read_parquet(INPUT_DATA_DIR / "merged_data_10282025_with_predictions.parquet",
                       engine="fastparquet")
data = data[data["session_id"] == "ses-00A"] # only include baseline data
print(data.shape)

(8768, 4325)


data.shape -> (8768, 4325)

In [5]:
# Filter to only keep specific columns along with all tracts/metrics
def filter_columns_by_tracts_and_metrics(columns):
    selected_cols = ["subject_session", "subject_id", "session_id", "age", "site", "sex", "qc_prediction",
                     "estimated_brain_volume", "scanner_model", "scanner_manufacturer"]
    for col in columns:
        parts = col.split('_')
        if parts[0] == 'msmt': selected_cols.append(col) # white matter feature columns
    return selected_cols

selected_columns = filter_columns_by_tracts_and_metrics(data.columns)
data_reduced = data[selected_columns].copy()
print(data_reduced.shape)

(8768, 4097)


data_reduced.shape -> (8768, 4097)

### ABCD Reproducible Matched Samples
Demographically matched samples.
We will perform 2-fold cross-validation with the matched samples serving as either the training or testing set in different folds.

In [6]:
def clean_coded_column(df, col, valid_range=None, missing_codes=(777, 999)):
    s = pd.to_numeric(df[col])               # converts numeric strings to numerics
    s = s.replace({code: np.nan for code in missing_codes})  # set missing codes to NaN
    lo, hi = valid_range
    s = s.where((s >= lo) & (s <= hi), np.nan) # if not in valid range, set to NaN
    df[col] = s
    return df

In [8]:
edu_sensitivity = False ## sensitivity analysis, looking at income and parental education rather than exposome score
cognition_analysis = True

In [9]:
participants = pd.read_csv(INPUT_DATA_DIR / "participants.tsv", sep="\t")
participants = participants.loc[participants["session_id"] == "ses-baselineYear1Arm1"].copy() # baseline arm only

# make subject id formatting consistent across datasets
participants.loc[:, "subject_id_clean"] = participants["participant_id"].str.replace(r"^sub-NDARINV", "", regex=True) 
data_reduced.loc[:, "subject_id_clean"] = data_reduced["subject_id"].str.replace(r"^sub-", "", regex=True)

if not edu_sensitivity:
    matched_info = participants.loc[:, ["subject_id_clean", "matched_group"]].drop_duplicates()
    data_reduced.loc[:, "matched_group"] = data_reduced["subject_id_clean"].map(matched_info.set_index("subject_id_clean")["matched_group"])
else: 
    participants = clean_coded_column(participants, col="parental_education", valid_range=(0, 21), missing_codes=(777, 999))
    participants = clean_coded_column(participants, col="income", valid_range=(1, 9), missing_codes=(777, 999))
    
    # drop participants with missing data and those who responded I dont know or prefer not to answer.
    participants_clean = participants.dropna(subset=["parental_education", "income"]).copy()
    
    cols_to_transfer = ["matched_group", "parental_education", "income"] # which cols from `participants` do we want in our final df
    
    lookup = (participants_clean.loc[:, ["subject_id_clean"] + cols_to_transfer].drop_duplicates(subset=["subject_id_clean"])
              .set_index("subject_id_clean"))
    
    for col in cols_to_transfer: data_reduced[col] = data_reduced["subject_id_clean"].map(lookup[col])

print(data_reduced["matched_group"].value_counts(dropna=False))
print(data_reduced.shape)
data_reduced.head(3)

matched_group
2.0      4297
1.0      4247
3.0       221
NaN         2
888.0       1
Name: count, dtype: int64
(8768, 4099)


,subject_session,subject_id,session_id,age,site,sex,qc_prediction,estimated_brain_volume,scanner_model,scanner_manufacturer,...,msmt_CerebellumInferiorCerebellarPeduncleL_NODDI_directions,msmt_CerebellumInferiorCerebellarPeduncleL_NODDI_icvf,msmt_CerebellumInferiorCerebellarPeduncleL_NODDI_isovf,msmt_CerebellumInferiorCerebellarPeduncleL_NODDI_od,msmt_CerebellumCerebellumL_NODDI_directions,msmt_CerebellumCerebellumL_NODDI_icvf,msmt_CerebellumCerebellumL_NODDI_isovf,msmt_CerebellumCerebellumL_NODDI_od,subject_id_clean,matched_group
0,sub-005V6D2C_ses-00A,sub-005V6D2C,ses-00A,10.1178082191781,10,2.0,0.593441,1327331.0,DISCOVERY MR750,GE MEDICAL SYSTEMS,...,-0.031898,0.576089,0.224058,0.395948,-0.147212,0.510994,0.206131,0.496900,005V6D2C,2.0
2,sub-007W6H7B_ses-00A,sub-007W6H7B,ses-00A,10.4986301369863,22,1.0,0.706109,1841484.0,DISCOVERY MR750,GE MEDICAL SYSTEMS,...,-0.065473,0.551795,0.251139,0.384457,-0.113333,0.484205,0.213963,0.458577,007W6H7B,1.0
4,sub-00CY2MDM_ses-00A,sub-00CY2MDM,ses-00A,10.8876712328767,20,1.0,0.878186,1470506.0,Prisma,SIEMENS,...,-0.114621,0.589538,0.227975,0.401330,-0.263468,0.540469,0.203819,0.455240,00CY2MDM,1.0


main analysis:
* shape = (8768, 4099)
* matched_group:  
    * 2.0:      4297
    * 1.0:      4247
    * 3.0:       221
    * NaN:         2
    * 888.0:       1
 
edu_sensitivity:
* shape = (8768, 4101)
* matched_group
    * 2.0:    3487
    * 1.0:    3420
    * NaN:    1680
    * 3.0:     181
      
cognition_analysis: 
* shape = (8768, 4099)
* matched_group
    * 2.0      4297
    * 1.0      4247
    * 3.0       221
    * NaN         2
    * 888.0       1

In [10]:
data_reduced = data_reduced.dropna(subset=["matched_group"])
data_reduced = data_reduced[data_reduced["matched_group"] != 888]
data_reduced = data_reduced[data_reduced["matched_group"] != 3]

print(data_reduced["matched_group"].value_counts(dropna=False))

matched_group
2.0    4297
1.0    4247
Name: count, dtype: int64


In [11]:
data_reduced['scanner_manufacturer'], manufacturer_mapping = pd.factorize(data_reduced['scanner_manufacturer'])
data_reduced['scanner_model'], model_mapping = pd.factorize(data_reduced['scanner_model'])
data_reduced_clean = (data_reduced.sort_values('subject_id_clean').drop_duplicates('subject_id_clean', keep='first'))
print(data_reduced.shape)

(8544, 4099)


main_analysis: (8544, 4099)

edu_sensitivity: (6907, 4101)

cognition_analysis: (8544, 4099)

### Exposome scores

In [12]:
exposome = pd.read_csv(INPUT_DATA_DIR / "abcd_longitudinal_factor_scores_7f_bifactor.csv")
exposome['subject_id_clean'] = exposome['ID'].str.replace(r'^NDAR_INV', '', regex=True)
print(exposome.shape)

(33372, 13)


In [13]:
exposome_filtered = exposome[exposome["Timepoint"] == "baseline_year_1_arm_1"]
exposome_filtered = exposome_filtered[["General_SES", "School", "Family_Values", "Family_Turmoil", 
                                       "Dense_Urban_Poverty", "Extracurriculars", "Screen_Time", "subject_id_clean"]]
print(exposome_filtered.shape)

(11874, 8)


### Cognition scores

In [14]:
cognition_data = pd.read_csv(INPUT_DATA_DIR / "cognition_data.csv")
cognition_data_filtered = cognition_data[cognition_data["event_name"] == "baseline_year_1_arm_1"]
cognition_core = cognition_data_filtered[["src_subject_id", "neurocog_pc1.bl", "neurocog_pc2.bl", "neurocog_pc3.bl"]]
cognition_core.loc[:, 'subject_id_clean'] = cognition_core['src_subject_id'].str.replace(r'^NDAR_INV', '', regex=True)
print(cognition_core.shape)
cognition_core.head(1)

(11878, 5)


/var/folders/9p/r96vjxyn44ddyjfz16lchms80000gp/T/ipykernel_26291/1127279331.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cognition_core.loc[:, 'subject_id_clean'] = cognition_core['src_subject_id'].str.replace(r'^NDAR_INV', '', regex=True)


,src_subject_id,neurocog_pc1.bl,neurocog_pc2.bl,neurocog_pc3.bl,subject_id_clean
1,NDAR_INV003RTV85,0.278102,0.340003,0.237957,003RTV85


### Drop bundles and metrics that we are not interested in

In [15]:
bundles_to_remove = [
    "msmt_ProjectionBrainstemDentatorubrothalamicTract-lr",
    "msmt_ProjectionBrainstemDentatorubrothalamicTract-rl",
    "msmt_CommissureAnteriorCommissure",
    "msmt_ProjectionBrainstemCorticobulbarTractL",
    "msmt_ProjectionBrainstemCorticobulbarTractR"
]

metrics_to_remove = ["number_of_tracts", "NODDI_directions"]

# --- Initial merge ---
if edu_sensitivity:
    merged_exposome = data_reduced.copy()
    print(f"[INIT] Using data_reduced only → shape: {merged_exposome.shape}")
else:
    merged_exposome = exposome_filtered.merge(
        data_reduced, on="subject_id_clean", how="outer"
    )
    print(f"[INIT] After merging exposome_filtered + data_reduced → shape: {merged_exposome.shape}")

# --- Cognition merge ---
if cognition_analysis:
    merged_exposome = merged_exposome.merge(
        cognition_core, on="subject_id_clean", how="outer"
    )
    print(f"[COGNITION MERGE] After adding cognition_core → shape: {merged_exposome.shape}")

# --- Drop bundle columns ---
cols_to_drop = [
    col for col in merged_exposome.columns
    if any(bundle in col for bundle in bundles_to_remove)
]
print(f"[DROP BUNDLES] Removing {len(cols_to_drop)} columns")
merged_exposome = merged_exposome.drop(columns=cols_to_drop)
print(f"[DROP BUNDLES] New shape → {merged_exposome.shape}")

# --- Drop metric columns ---
cols_to_drop = [
    col for col in merged_exposome.columns
    if any(metric in col for metric in metrics_to_remove)
]
print(f"[DROP METRICS] Removing {len(cols_to_drop)} columns")
merged_exposome = merged_exposome.drop(columns=cols_to_drop)
print(f"[DROP METRICS] New shape → {merged_exposome.shape}")

# --- Drop NA ---
before_rows = merged_exposome.shape[0]
merged_exposome = merged_exposome.dropna()
after_rows = merged_exposome.shape[0]

print(f"[DROP NA] Rows before: {before_rows}, after: {after_rows}, removed: {before_rows - after_rows}")
print(f"[FINAL] Final shape → {merged_exposome.shape}")

# Preview
merged_exposome.head(2)

[INIT] After merging exposome_filtered + data_reduced → shape: (11876, 4106)
[COGNITION MERGE] After adding cognition_core → shape: (11878, 4110)
[DROP BUNDLES] Removing 305 columns
[DROP BUNDLES] New shape → (11878, 3805)
[DROP METRICS] Removing 124 columns
[DROP METRICS] New shape → (11878, 3681)
[DROP NA] Rows before: 11878, after: 7877, removed: 4001
[FINAL] Final shape → (7877, 3681)


,General_SES,School,Family_Values,Family_Turmoil,Dense_Urban_Poverty,Extracurriculars,Screen_Time,subject_id_clean,subject_session,subject_id,...,msmt_CerebellumInferiorCerebellarPeduncleL_NODDI_isovf,msmt_CerebellumInferiorCerebellarPeduncleL_NODDI_od,msmt_CerebellumCerebellumL_NODDI_icvf,msmt_CerebellumCerebellumL_NODDI_isovf,msmt_CerebellumCerebellumL_NODDI_od,matched_group,src_subject_id,neurocog_pc1.bl,neurocog_pc2.bl,neurocog_pc3.bl
1,-0.975790,1.003044,0.429337,-1.085106,1.752441,-0.945097,-1.305787,005V6D2C,sub-005V6D2C_ses-00A,sub-005V6D2C,...,0.224058,0.395948,0.510994,0.206131,0.496900,2.0,NDAR_INV005V6D2C,-0.930358,0.070785,-0.047210
2,1.362875,-1.674093,0.646487,0.532428,2.151757,0.214627,-1.044261,007W6H7B,sub-007W6H7B_ses-00A,sub-007W6H7B,...,0.251139,0.384457,0.484205,0.213963,0.458577,1.0,NDAR_INV007W6H7B,0.983973,0.098231,0.797205


**main_analysis**
* [INIT] After merging exposome_filtered + data_reduced → shape: (11876, 4106)
* [DROP BUNDLES] Removing 305 columns
* [DROP BUNDLES] New shape → (11876, 3801)
* [DROP METRICS] Removing 124 columns
* [DROP METRICS] New shape → (11876, 3677)
* [DROP NA] Rows before: 11876, after: 8428, removed: 3448
* [FINAL] Final shape → (8428, 3677)

**edu_sensitivity**
* [INIT] Using data_reduced only → shape: (6907, 4101)
* [DROP BUNDLES] Removing 305 columns
* [DROP BUNDLES] New shape → (6907, 3796)
* [DROP METRICS] Removing 124 columns
* [DROP METRICS] New shape → (6907, 3672)
* [DROP NA] Rows before: 6907, after: 6809, removed: 98
* [FINAL] Final shape → (6809, 3672)

**cognition_analysis**
* [INIT] After merging exposome_filtered + data_reduced → shape: (11876, 4106)
* [COGNITION MERGE] After adding cognition_core → shape: (11878, 4110)
* [DROP BUNDLES] Removing 305 columns
* [DROP BUNDLES] New shape → (11878, 3805)
* [DROP METRICS] Removing 124 columns
* [DROP METRICS] New shape → (11878, 3681)
* [DROP NA] Rows before: 11878, after: 7877, removed: 4001
* [FINAL] Final shape → (7877, 3681)

In [16]:
OUTPUT_DIR = ROOT_DIR / "output_data" / "cleaned"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)  # create if it doesn't exist

if edu_sensitivity:
    output_file = OUTPUT_DIR / "parental_edu_income_sensitivity_df.csv"
elif cognition_analysis:
    output_file = OUTPUT_DIR / "exposome_FINAL_cognition_11_10.csv"
else:
    output_file = OUTPUT_DIR / "exposome_FINAL_11_3.csv"

merged_exposome.to_csv(output_file, index=False)

print(f"[SAVE] File saved to: {output_file}")

[SAVE] File saved to: /Users/bmacedo/Desktop/final_WM/output_data/cleaned/exposome_FINAL_cognition_11_10.csv
